In [1]:
import yaml


mapping = {
    'activity_col_write': 0.5,
    'activity_row_write': 0.5,
    'activity_col_read': 0.5,
    'activity_row_read': 0.5,
    'num_read_cell_op': 64,
    'num_write_cell_op': 64,
}

with open('../../mapping.yaml', 'w') as file:
    yaml.dump(mapping, file, default_flow_style=False)


In [2]:
with open('../../mapping.yaml', 'r') as file:
    loaded_config = yaml.safe_load(file)


print(loaded_config)


{'activity_col_read': 0.5, 'activity_col_write': 0.5, 'activity_row_read': 0.5, 'activity_row_write': 0.5, 'num_read_cell_op': 64, 'num_write_cell_op': 64}


In [20]:
import math
import sys
import yaml
import numpy as np
import csv
import torch
import os
sys.path.append('../../python/')  
from periphery import logicGate
from periphery import constant
from periphery.Technology import Technology

def generate_mapping_yaml(
    filepath='param.yaml',
    # cell config
    cell_type: str = 'SRAM',
    accesstype: str = 'CMOS_access',   # 'CMOS_access' or 'crossbar'
    # weight sources (choose ONE)
    weights: np.ndarray | None = None,     # direct numpy array [rows, cols]
    pth_path: str | None = None,           # pass .pth file and layer name
    layer_name: str | None = None,
    csv_path: str | None = None,           # cvs file path
    # quantization/mapping params
    synapse_bit: int = 8,                  # quantization bits
    cell_bit: int = 1,                     # bits per cell
    algo_weight_max: float = 1.0,          # algorithm upper weight bound
    algo_weight_min: float = -1.0,         # algorithm lower weight bound
    max_conductance: float = 25e-6,        # conductance mapping upper bound (S)
    min_conductance: float = 5e-6,         # conductance mapping lower bound (S)
    # outputs
    save_slice_path = 'slice.npy',
    save_conductance_path = 'conductance.npy'
):
    
    # ---------- choose and load the weight ----------
    raw_weights = None
    

    if weights is not None:
        raw_weights = np.asarray(weights, dtype=float)

    elif pth_path is not None and layer_name is not None:
        
        ckpt = torch.load(pth_path, map_location='cpu')
        # auto-detect state_dict format
        state_dict = ckpt if isinstance(ckpt, dict) and any(k.endswith('.weight') or k.endswith('.bias') for k in ckpt.keys()) \
                     else ckpt.get('state_dict', {})
        if layer_name not in state_dict:
            raise KeyError(f"cannot find layer '{layer_name}' in the state_dict keys.")
        raw_weights = state_dict[layer_name].detach().cpu().numpy().astype(float)

    elif csv_path is not None:
        rows_weight = []
        with open(csv_path, 'r', newline='') as f:
            reader = csv.reader(f)
            for row in reader:
                if len(row) == 0:
                    continue
                rows_weight.append([float(x) for x in row])
        if not rows_weight:
            raise ValueError("CSV is empty or has no valid rows.")
        max_len = max(len(r) for r in rows_weight)
        rows_weight = [r + [0.0]*(max_len - len(r)) for r in rows_weight]
        raw_weights = np.array(rows_weight, dtype=float)

    else:
        raise ValueError("please provide ONE weight source: weights ndarray, or pth_path & layer_name, or csv_path.")

    if raw_weights.ndim != 2:
        raise ValueError(f"weights should be 2D array, got shape {raw_weights.shape}.")

    rows_weight, cols_weight = raw_weights.shape

    num_col_per_synapse = synapse_bit // cell_bit


    quantized_min = 0.0
    quantized_max = 2. ** synapse_bit - 1.
    cell_range = 2 ** cell_bit  # range of a single cell

    # ---------- SRAM maping：quantization -> bit-slice -> conductance ----------
    # size of output matrix：[rows, cols * num_col_per_synapse]
    out_cols = cols_weight * num_col_per_synapse
    slice_mat = np.zeros((rows_weight, out_cols), dtype=int)
    conductance_mat = np.zeros((rows_weight, out_cols), dtype=float)

    for r in range(rows_weight):
        out_c = 0
        for c in range(cols_weight):
            raw_weight = raw_weights[r, c]

            scale = (algo_weight_max - algo_weight_min) / (quantized_max - quantized_min)
            zero_point = quantized_max - algo_weight_max / scale
            if zero_point < quantized_min:
                zero_point = torch.tensor([quantized_min], dtype=torch.float32).to(algo_weight_min.device)
            elif zero_point > quantized_max:
                # zero_point = qmax
                zero_point = torch.tensor([quantized_max], dtype=torch.float32).to(algo_weight_max.device)
            zero_point = np.round(zero_point)
            quantized_weight = zero_point + raw_weight / scale
            zero_point = np.clip(quantized_min, quantized_min, quantized_max)
            zero_point = np.round(zero_point)



            # bit-slice（msb---lsb）
            slice_vals = []
            value = quantized_weight
            for _ in range(num_col_per_synapse):
                remainder = int(value % cell_range)
                value = int(value // cell_range)
                slice_vals.insert(0, remainder)

            # linear mapping to conductance
            for sv in slice_vals:
                # sv ∈ [0, cell_range-1] → [minG, maxG]
                # notice：sv=0 → min_conductance (not zero!)
                conduct = (sv / (cell_range - 1.0)) * (max_conductance - min_conductance) + min_conductance
                conductance_mat[r, out_c] = conduct
                slice_mat[r, out_c] = sv
                out_c += 1

    # ---------- save the conductance ----------
    os.makedirs(os.path.dirname(save_conductance_path) or ".", exist_ok=True)
    np.save(save_conductance_path, conductance_mat)
    np.save(save_slice_path, slice_mat)
    
    

    # Create mapping dictionary
    mapping = {
    'activity_col_write': 0.5,
    'activity_row_write': 0.5,
    'activity_col_read': 0.5,
    'activity_row_read': 0.5,
    'num_read_cell_op': 64,
    'num_write_cell_op': 64,
    'numRowParallel': 1,
    'numColMuxed': 1,

    'shape_algo': [int(rows_weight), int(cols_weight)],
    'synapse_bit': int(synapse_bit),
    'cell_bit': int(cell_bit),
    'num_col_per_synapse': int(num_col_per_synapse),
    'algo_weight_min': float(algo_weight_min),
    'algo_weight_max': float(algo_weight_max),
    'min_conductance': float(min_conductance),
    'max_conductance': float(max_conductance),
    'slice_path': save_slice_path,
    'conductance_path': save_conductance_path
}
    # Write to YAML
    with open(filepath, 'w') as file:
        yaml.dump(mapping, file, default_flow_style=False)

    print(f"param.yaml generated at: {filepath}")
    return mapping


In [15]:
fake_weights = np.random.uniform(-1.0, 1.0, (84, 10))

In [23]:
generate_mapping_yaml(
    filepath='../../mapping.yaml',
    # cell config
    cell_type = 'SRAM',
    accesstype  = 'CMOS_access',   # 'CMOS_access' or 'crossbar'
    # weight sources (choose ONE)
    weights = fake_weights,     # direct numpy array [rows, cols]
    pth_path = None,           # pass .pth file and layer name
    layer_name = None,
    csv_path = None,           # cvs file path
    # quantization/mapping params
    synapse_bit = 8,                  # quantization bits
    cell_bit = 1,                     # bits per cell
    algo_weight_max  = 1.0,          # algorithm upper weight bound
    algo_weight_min  = -1.0,         # algorithm lower weight bound
    max_conductance  = 25e-6,        # conductance mapping upper bound (S)
    min_conductance  = 5e-6,         # conductance mapping lower bound (S)
    # outputs
    save_slice_path = '../../slice.npy',
    save_conductance_path = '../../conductance.npy'
)

param.yaml generated at: ../../mapping.yaml


{'activity_col_write': 0.5,
 'activity_row_write': 0.5,
 'activity_col_read': 0.5,
 'activity_row_read': 0.5,
 'num_read_cell_op': 64,
 'num_write_cell_op': 64,
 'numRowParallel': 1,
 'numColMuxed': 1,
 'shape_algo': [84, 10],
 'synapse_bit': 8,
 'cell_bit': 1,
 'num_col_per_synapse': 8,
 'algo_weight_min': -1.0,
 'algo_weight_max': 1.0,
 'min_conductance': 5e-06,
 'max_conductance': 2.5e-05,
 'slice_path': '../../slice.npy',
 'conductance_path': '../../conductance.npy'}

In [17]:
#check slice npy file
loaded_slice = np.load('../../slice.npy')
print("Loaded slice shape:", loaded_slice.shape)
print("Loaded slice data (first 5 rows):", loaded_slice[:5])
#check conductance npy file

Loaded slice shape: (84, 80)
Loaded slice data (first 5 rows): [[1 0 1 0 0 0 0 0 0 1 1 0 1 0 1 0 1 1 0 0 0 0 1 1 0 0 0 1 0 1 1 1 1 1 0 1
  1 0 1 1 0 1 0 1 0 1 0 1 1 1 0 0 1 1 0 1 1 1 1 1 0 0 1 0 0 0 1 0 1 1 0 1
  0 1 0 1 0 0 0 1]
 [1 1 0 0 0 1 1 1 0 0 0 1 0 0 1 1 1 0 1 0 1 1 1 0 1 0 1 1 0 1 1 1 0 0 0 0
  0 1 0 0 1 0 0 0 0 1 0 0 1 1 1 0 0 0 0 1 1 0 1 1 1 0 1 0 1 1 1 1 0 0 0 0
  1 1 0 1 0 0 1 1]
 [1 1 1 0 0 0 1 1 0 1 0 1 1 0 0 0 0 1 1 0 1 0 0 0 1 1 0 0 0 1 0 1 0 1 0 1
  0 1 1 0 1 1 1 0 0 0 0 0 0 1 1 0 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 1 0 1 1
  0 0 1 1 0 1 0 1]
 [1 1 0 1 1 0 1 1 1 0 0 1 1 1 1 0 0 0 0 1 1 0 0 0 1 1 1 0 1 1 0 0 1 1 1 0
  1 0 1 1 0 1 1 0 1 0 0 1 1 0 1 0 1 0 0 1 0 0 1 0 0 0 1 0 0 1 1 0 1 0 1 1
  0 1 0 1 0 1 1 1]
 [0 1 1 1 1 0 0 0 1 1 1 0 0 0 0 1 1 0 0 0 0 1 1 1 0 1 0 0 1 0 0 0 0 0 1 0
  1 1 1 0 0 0 0 0 1 0 0 1 0 0 1 1 1 1 0 0 0 1 1 0 1 0 0 0 0 1 0 1 0 0 0 1
  0 1 0 0 0 1 0 1]]
